# 第十四章：MLOps 与模型部署

## 从 Jupyter Notebook 到生产环境

训练出一个好模型只是 AI 工程的 20%。**MLOps** 解决的是剩下的 80%：部署、监控、迭代、扩展。

```
训练完成 → 模型导出 → 服务化封装 → 容器化 → 部署上线 → 监控 → 反馈迭代
```

## 14.1 模型导出与优化

### 导出格式

| 格式 | 特点 | 使用场景 |
|------|------|---------|
| **PyTorch .pt** | 完整模型，灵活 | 继续训练、研究 |
| **TorchScript** | Python-free 序列化 | C++ 部署 |
| **ONNX** | 跨框架标准格式 | 多框架部署 |
| **GGUF** | llama.cpp 量化格式 | LLM 本地推理 |
| **TensorRT** | NVIDIA 推理优化 | GPU 高性能推理 |

### 模型量化

将 FP32 参数压缩为 INT8/INT4，大幅减少显存和延迟：

| 精度 | 相对大小 | 精度损失 | 场景 |
|------|---------|---------|------|
| FP32 | 100% | 0 | 训练 |
| FP16 | 50% | ~0% | 推理 |
| INT8 | 25% | < 0.5% | 生产推理 |
| INT4 | 12.5% | 1-3% | 边缘设备 |


In [ ]:
# === ONNX 导出示例 ===
import torch

class SimpleModel(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = torch.nn.Linear(10, 2)
    
    def forward(self, x):
        return self.fc(x)

model = SimpleModel()
model.eval()

# 导出为 ONNX
dummy_input = torch.randn(1, 10)
torch.onnx.export(
    model, dummy_input, "model.onnx",
    input_names=["input"],
    output_names=["output"],
    dynamic_axes={"input": {0: "batch"}, "output": {0: "batch"}}
)
# 生成 model.onnx，可用 ONNX Runtime 或 TensorRT 推理


## 14.2 模型服务化 (Model Serving)

### FastAPI 模型服务

```python
from fastapi import FastAPI
from pydantic import BaseModel
import torch

app = FastAPI()
model = torch.load("model.pt")
model.eval()

class PredictRequest(BaseModel):
    features: list[float]

class PredictResponse(BaseModel):
    prediction: int
    confidence: float

@app.post("/predict", response_model=PredictResponse)
def predict(req: PredictRequest):
    with torch.no_grad():
        x = torch.tensor(req.features).unsqueeze(0)
        logits = model(x)
        prob = torch.softmax(logits, dim=-1)
        pred = torch.argmax(prob, dim=-1).item()
    return PredictResponse(prediction=pred, confidence=prob[0,pred].item())
```

### 关键考量

| 考量 | 策略 |
|------|------|
| **批量推理** | 攒够 N 个请求或超时后批量处理，GPU 利用率翻倍 |
| **动态批处理** | Triton Inference Server 内置支持 |
| **模型版本管理** | MLflow / DVC 追踪实验和模型版本 |
| **A/B Testing** | 流量分流到不同模型版本，对比效果 |

## 14.3 容器化与编排

### Docker 部署

```dockerfile
FROM python:3.11-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY model.pt app.py .
EXPOSE 8000
CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]
```

### GPU 支持

NVIDIA Container Toolkit 使 Docker 容器可访问宿主机 GPU：

```bash
docker run --gpus all -p 8000:8000 my-model:latest
```

### LLM 推理引擎

| 引擎 | 特点 |
|------|------|
| **vLLM** | PagedAttention，高吞吐量，连续批处理 |
| **Ollama** | 一键本地部署，量化模型，REST API |
| **llama.cpp** | C++ 推理，GGUF 量化，CPU 友好 |
| **Text Generation Inference (TGI)** | HuggingFace 官方，生产就绪 |

## 14.4 监控与迭代

生产环境模型需要持续监控：

| 监控对象 | 指标 | 告警阈值 |
|---------|------|---------|
| **预测延迟** | P50/P95/P99 | P95 > 500ms |
| **吞吐量** | QPS (每秒请求数) | 低于预期 50% |
| **数据漂移** | 输入分布变化 | PSI > 0.2 |
| **模型衰减** | 准确率下降 | 相对下降 > 5% |

### 反馈循环

```
用户反馈 (👍/👎) → 收集正负样本 → 定期微调 → A/B 测试 → 上线新版本
```

## 本章小结

| 阶段 | 工具/技术 |
|------|----------|
| **导出** | ONNX, TorchScript, GGUF |
| **量化** | INT8/INT4, GPTQ, AWQ |
| **服务** | FastAPI, Triton, vLLM |
| **容器** | Docker, NVIDIA Container Toolkit |
| **监控** | Prometheus, Grafana, 数据漂移检测 |